<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/MinhKhoa/winequality-red.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV

In [2]:
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/conquer1/AIO-Conquer-Data/Data/dataset/winequality-red.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tải dữ liệu thành công! Kích thước: (1599, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  


In [3]:
target_colum = 'quality'
X = df.drop(columns=[target_colum], errors='ignore')
y = df[target_colum]

# 2. Chia tập Train/Test trước để tránh data leakage khi tính trung bình giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
lr_model = LinearRegression()

# 2. Huấn luyện mô hình trên tập dữ liệu đã encode
lr_model.fit(X_train, y_train)

# 3. Tiến hành dự đoán cân nặng
preds = lr_model.predict(X_test)
predtrain = lr_model.predict(X_train)

# 4. Đánh giá mô hình bằng R2-score, RMSE, MAE
r2_score_all = round(r2_score(y_test, preds), 3)
r2_score_train = round(r2_score(y_train, predtrain), 3)
rmse_all = round(np.sqrt(mean_squared_error(y_test, preds)), 3)
mae_all = round(mean_absolute_error(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng Linear Regression: {r2_score_all}")
print(f"Chỉ số R2-score của train khi sử dụng Linear Regression: {r2_score_train}")
print(f"Chỉ số RMSE khi sử dụng Linear Regression: {rmse_all}")
print(f"Chỉ số MAE khi sử dụng Linear Regression: {mae_all}")

Chỉ số R2-score khi sử dụng Linear Regression: 0.403
Chỉ số RMSE khi sử dụng Linear Regression: 0.625
Chỉ số MAE khi sử dụng Linear Regression: 0.504


# **1.Boruta**

In [5]:
# --- 1. Cấu hình và chạy Boruta ---
# BorutaPy yêu cầu mảng NumPy (.values) để tránh lỗi index
X_train_np = X_train.values
y_train_np = y_train.values

# Giữ nguyên Random Forest làm mô hình nền để Boruta tính toán feature_importances_
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
boruta_selector = BorutaPy(estimator=rf, n_estimators='auto', verbose=0, random_state=42, max_iter=100)
boruta_selector.fit(X_train_np, y_train_np)

# --- 2. Lọc và huấn luyện lại mô hình ---
# Lọc lấy các tính năng quan trọng được Boruta xác nhận
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test.values)

#Khởi tạo và huấn luyện mô hình Linear Regression trên tập dữ liệu đã lọc
lr_final = LinearRegression()
lr_final.fit(X_train_filtered, y_train_np)

# --- 3. Dự đoán và đánh giá điểm R2 ---
# Sử dụng mô hình Linear Regression vừa train để dự đoán
preds_boruta = lr_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test.values, preds_boruta), 3)
rsme_boruta = round(np.sqrt(mean_squared_error(y_test.values, preds_boruta)), 3)
mae_boruta = round(mean_absolute_error(y_test.values, preds_boruta), 3)

# --- 4. Trích xuất danh sách tính năng được chọn ---
feature_names = X_train.columns
confirmed_features = feature_names[boruta_selector.support_].tolist()

# --- 5. Xuất kết quả báo cáo ---
print(f"{'KẾT QUẢ SÀN LỌC TÍNH NĂNG & HUẤN LUYỆN LINEAR REGRESSION':^60}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_boruta}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rsme_boruta}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_boruta}")
print(f"• Số lượng tính năng được giữ lại   : {len(confirmed_features)}/{len(feature_names)}")
print(f"• Các tính năng được giữ lại:\n  {confirmed_features}")
print("="*60)

  KẾT QUẢ SÀN LỌC TÍNH NĂNG & HUẤN LUYỆN LINEAR REGRESSION  
• Chỉ số R2-score (Linear Regression) : 0.404
• Chỉ số RMSE (Linear Regression)    : 0.624
• Chỉ số MAE (Linear Regression)     : 0.503
• Số lượng tính năng được giữ lại   : 9/11
• Các tính năng được giữ lại:
  ['fixed acidity', 'volatile acidity', 'residual sugar', 'chlorides', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']


# **2.LASSO**

In [6]:
# --- 1. Chuẩn hóa dữ liệu (Feature Scaling) ---
# Bước này bắt buộc phải có vì Lasso rất nhạy cảm với scale của dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2. Khởi tạo và chạy LassoCV để SÀNG LỌC TÍNH NĂNG ---
lasso = LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, random_state=42, n_jobs=-1)
lasso.fit(X_train_scaled, y_train)

# Trích xuất danh sách tên các tính năng được Lasso giữ lại (Coef != 0)
feature_names = X_train.columns
kept_features = feature_names[lasso.coef_ != 0].tolist()
dropped_features = feature_names[lasso.coef_ == 0].tolist()

# --- 3. Lọc lại dữ liệu theo các tính năng được chọn ---
# Lấy vị trí index của các tính năng được giữ lại để lọc trên mảng numpy đã scaled
kept_features_idx = [X_train.columns.get_loc(col) for col in kept_features]
X_train_filtered = X_train_scaled[:, kept_features_idx]
X_test_filtered = X_test_scaled[:, kept_features_idx]

# --- 4. Huấn luyện mô hình LINEAR REGRESSION trên các tính năng đã lọc ---
lr_final = LinearRegression(n_jobs=-1)
lr_final.fit(X_train_filtered, y_train)

# --- 5. Dự đoán và tính toán các chỉ số đánh giá ---
preds_lr = lr_final.predict(X_test_filtered)
r2_score_lr = round(r2_score(y_test, preds_lr), 3)
rmse_lr = round(np.sqrt(mean_squared_error(y_test, preds_lr)), 3)
mae_lr = round(mean_absolute_error(y_test, preds_lr), 3)

# --- 6. Xuất kết quả báo cáo ---
print("="*60)
print(f"{'KẾT QUẢ SÀN LỌC BẰNG LASSO & HUẤN LUYỆN LINEAR REGRESSION':^60}")
print("="*60)
print(f"• Alpha tối ưu của Lasso             : {round(lasso.alpha_, 5)}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_lr}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rmse_lr}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_lr}")
print(f"• Số lượng tính năng bị loại bỏ      : {len(dropped_features)}/{len(feature_names)}")
print(f"• Số lượng tính năng được giữ lại    : {len(kept_features)}/{len(feature_names)}")
print("-"*60)
if dropped_features:
    print(f"• Các tính năng bị loại bỏ (Lasso Coef = 0):\n  {dropped_features}\n")
print(f"• Các tính năng được chọn để chạy Linear Regression:\n  {kept_features}")
print("="*60)

 KẾT QUẢ SÀN LỌC BẰNG LASSO & HUẤN LUYỆN LINEAR REGRESSION  
• Alpha tối ưu của Lasso             : 0.0083
• Chỉ số R2-score (Linear Regression) : 0.401
• Chỉ số RMSE (Linear Regression)    : 0.626
• Chỉ số MAE (Linear Regression)     : 0.504
• Số lượng tính năng bị loại bỏ      : 3/11
• Số lượng tính năng được giữ lại    : 8/11
------------------------------------------------------------
• Các tính năng bị loại bỏ (Lasso Coef = 0):
  ['citric acid', 'residual sugar', 'density']

• Các tính năng được chọn để chạy Linear Regression:
  ['fixed acidity', 'volatile acidity', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'pH', 'sulphates', 'alcohol']
